# Kiểm thử API RAGProvider (Graph Database Legal Assistant)

- **Thư mục root**: `ML_final`
- **Mô hình Graph**: Local Graph Database (`db/graph_database`)
- **Hỗ trợ 2 chế độ**: `rag=True` (w/ RAG) và `rag=False` (w/o RAG - Direct LLM)

Notebook hỗ trợ tự động: Nếu đã bật server qua `uvicorn` (port 8002) sẽ dùng HTTP Client để tiết kiệm VRAM GPU, nếu chưa bật sẽ nạp `TestClient` in-process.

In [1]:
import os
import sys
import json
from pathlib import Path
import httpx

# Đảm bảo ML_final nằm trong sys.path và là working directory
CURRENT_DIR = Path.cwd()
for p in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if p.name == "ML_final" or (p / "be" / "src").exists() or (p / "pyproject.toml").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        os.chdir(str(p))
        break

print(f"📁 Thư mục làm việc hiện tại: {Path.cwd()}")

# Tự động kết nối Live Server trên port 8002 nếu đang chạy (tránh nạp trùng model làm tràn 8GB VRAM)
client = None
try:
    with httpx.Client(base_url="http://localhost:8002", timeout=3.0) as check_c:
        resp = check_c.get("/health")
        if resp.status_code == 200:
            client = httpx.Client(base_url="http://localhost:8002", timeout=120.0)
            print("⚡ Đã kết nối tới Live Server RAGProvider (http://localhost:8002) - Tối ưu VRAM GPU!")
except Exception:
    pass

if client is None:
    from fastapi.testclient import TestClient
    from be.src.RAGProvider.api import app
    client = TestClient(app)
    print("✅ Khởi tạo TestClient RAGProvider (In-Process) thành công!")

📁 Thư mục làm việc hiện tại: c:\Users\VINH\OneDrive - VNU-HCMUS\Attachments\Desktop\ML_final
⚡ Đã kết nối tới Live Server RAGProvider (http://localhost:8002) - Tối ưu VRAM GPU!


## 1. Kiểm tra trạng thái hệ thống (`GET /health`)

In [2]:
res_health = client.get("/health")
print(f"Status Code: {res_health.status_code}")
print(json.dumps(res_health.json(), indent=2, ensure_ascii=False))

Status Code: 200
{
  "status": "healthy",
  "service": "RAGProvider",
  "retrieval_mode": "local_lancedb",
  "embedding_model": "BAAI/bge-m3",
  "reranker_model": "BAAI/bge-reranker-v2-m3",
  "llm_model": "gemma-4-26b-a4b-it",
  "nli_service_configured": true,
  "device": "cuda",
  "timestamp": "2026-08-29T12:30:32.502927+00:00"
}


## 2. Kiểm thử chế độ w/o RAG (`rag=False` -> Direct LLM Answer)

In [3]:
direct_payload = {
    "query": "Quyền sử dụng đất là gì?",
    "rag": False,
    "session_id": "test_direct_001",
    "metadata": {
        "user_id": "vinh_01"
    }
}

res_direct = client.post("/query", json=direct_payload)
print(f"Status Code: {res_direct.status_code}")
print(json.dumps(res_direct.json(), indent=2, ensure_ascii=False))

Status Code: 200
{
  "request_id": "1533167c-c424-4e0b-92cf-780a33bc16f4",
  "timestamp": "2026-08-29T12:30:32.513048+00:00",
  "query": "Quyền sử dụng đất là gì?",
  "rewritten_query": null,
  "rag_used": false,
  "retrieval_mode": null,
  "answer": "Chào bạn, với tư cách là một luật sư, tôi xin giải đáp thắc mắc của bạn về khái niệm **\"Quyền sử dụng đất\"** dựa trên các quy định của pháp luật Việt Nam hiện hành (Luật Đất đai) như sau:\n\n### 1. Bản chất pháp lý của Quyền sử dụng đất\nĐể hiểu đúng về quyền sử dụng đất, trước hết bạn cần nắm vững một nguyên tắc hiến định tại Việt Nam: **Đất đai thuộc sở hữu toàn dân do Nhà nước đại diện chủ sở hữu và thống nhất quản lý.**\n\nĐiều này có nghĩa là: **Không có cá nhân hay tổ chức nào có \"quyền sở hữu đất đai\"** theo nghĩa tuyệt đối như sở hữu một vật phẩm cá nhân. Thay vào đó, Nhà nước trao cho các cá nhân, hộ gia đình và tổ chức **\"quyền sử dụng đất\"**.\n\nVì vậy, **Quyền sử dụng đất (QSDĐ)** là một loại **quyền tài sản**. Nó là quy

## 3. Kiểm thử chế độ w/ RAG (`rag=True` -> Query Rewrite + Graph DB + Reranker + Enhanced LLM)

In [4]:
rag_payload = {
    "query": "Hạn mức giao đất nông nghiệp cho cá nhân theo luật đất đai 2024 quy định như thế nào?",
    "rag": True,
    "top_k": 3,
    "session_id": "test_rag_002",
    "metadata": {
        "user_id": "vinh_01"
    }
}

res_rag = client.post("/query", json=rag_payload)
print(f"Status Code: {res_rag.status_code}")
data = res_rag.json()
print(json.dumps(data, indent=2, ensure_ascii=False))

Status Code: 200
{
  "request_id": "1b960485-af39-44f9-ad63-4ebb3dc44ae0",
  "timestamp": "2026-08-29T12:31:14.379628+00:00",
  "query": "Hạn mức giao đất nông nghiệp cho cá nhân theo luật đất đai 2024 quy định như thế nào?",
  "rewritten_query": "Hạn mức giao đất nông nghiệp cho cá nhân theo Luật Đất đai 2024",
  "rag_used": true,
  "retrieval_mode": "local_lancedb",
  "answer": "Không có chứng cứ xác minh.",
  "nli_verification": {
    "is_valid": false,
    "label": "CONTRADICTION/LOSE",
    "confidence": 0.9994,
    "probabilities": {
      "CONTRADICTION/LOSE": 0.9994,
      "ENTAILMENT/WIN": 0.0006
    },
    "note": "Cảnh báo NLI: Mâu thuẫn/Không thỏa mãn (Contradiction/Lose)."
  },
  "retrieved_chunks": [
    {
      "id": "a16decbc8aa9222501be6911ceb14cf82b105c7e7d79ae337e422bb0985e8efdcd32fccd5522c2b6e1686c499be3ac8ea99e164d4ea39438f1b81a96c4550a05",
      "text": " khai thác tài nguyên nước, dầu khí hoặc loại khoáng sản khác 500.000.000 đồng trở lên;\n\nb) Khoáng sản trị giá

## 4. Kiểm thử endpoint tương thích (`POST /retrieve`)

In [5]:
retrieve_payload = {
    "query": "Điều kiện chuyển nhượng quyền sử dụng đất",
    "rag": True,
    "top_k": 2,
    "session_id": "test_retrieve_003"
}

res_retrieve = client.post("/retrieve", json=retrieve_payload)
print(f"Status Code: {res_retrieve.status_code}")
print(json.dumps(res_retrieve.json(), indent=2, ensure_ascii=False))

ReadTimeout: timed out